In [2]:
import ee
import geemap
import pandas as pd

# 1. Initialize Earth Engine
try:
    ee.Initialize(project='replicating-paper')
    print("Google Earth Engine Initialized successfully.")
except Exception as e:
    print("Authentication required...")
    ee.Authenticate()
    ee.Initialize(project='replicating-paper')
    print("Google Earth Engine Authenticated and Initialized.")

# 2. Define the Target and Timeline
ROI_NAME = 'Sanjay Van, Delhi'
roi = ee.Geometry.Rectangle([77.17, 28.52, 77.18, 28.54])
region= [77.17 - 0.01, 28.52 - 0.01, 77.18 + 0.01, 28.54 + 0.01]
roi = ee.Geometry.Rectangle(region)


# 5-Year Window for Robust Harmonic Math
START_DATE = '2020-01-01'
END_DATE = '2024-12-31'

print(f"\n--- ROI Setup ---")
print(f"Target: {ROI_NAME}")
print(f"Timeline: {START_DATE} to {END_DATE} (5 Years)")

# 3. Pre-Flight Check: Data Density (Valid Observation Count)
print("\nCalculating Valid Satellite Observations (Cloud < 20%)...")

# Pull the Sentinel-2 archive
s2_archive = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

# Count the number of valid images available for each pixel
observation_count = s2_archive.count().select('B4').rename('Valid_Count')

# Get the stats for our specific ROI
density_stats = observation_count.reduceRegion(
    reducer=ee.Reducer.min().combine(ee.Reducer.max(), sharedInputs=True).combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=roi,
    scale=10,
    maxPixels=1e9
).getInfo()

min_obs = density_stats.get('Valid_Count_min', 0)
max_obs = density_stats.get('Valid_Count_max', 0)
mean_obs = density_stats.get('Valid_Count_mean', 0)

print(f"\n--- Data Density Report ---")
print(f"Minimum observations on a pixel: {min_obs}")
print(f"Maximum observations on a pixel: {max_obs}")
print(f"Average observations per pixel: {mean_obs:.1f}")

if min_obs < 40:
    print("\n WARNING: Some pixels have very sparse data (<40 observations over 5 years).")
    print("These 'ghost' pixels will cause our math to hallucinate. We will drop them in Phase 1.")
else:
    print("\n DATA HEALTHY: All pixels have sufficient data for 5-year harmonic regression.")

# 4. Visual Diagnostic Map
Map = geemap.Map()
Map.centerObject(roi, 14)
Map.addLayer(roi, {'color': 'blue'}, "ROI Boundary", False)

# Visualize the observation count (Red = Sparse/Bad, Green = Dense/Good)
vis_params = {
    'min': 20,
    'max': 120,
    'palette': ['red', 'orange', 'yellow', 'green', 'darkgreen']
}
Map.addLayer(observation_count.clip(roi), vis_params, "Data Density (Valid Snapshots)")

Map

Google Earth Engine Initialized successfully.

--- ROI Setup ---
Target: Sanjay Van, Delhi
Timeline: 2020-01-01 to 2024-12-31 (5 Years)

Calculating Valid Satellite Observations (Cloud < 20%)...

--- Data Density Report ---
Minimum observations on a pixel: 190
Maximum observations on a pixel: 191
Average observations per pixel: 191.0

 DATA HEALTHY: All pixels have sufficient data for 5-year harmonic regression.


Map(center=[28.52999955882056, 77.17499999999988], controls=(WidgetControl(options=['position', 'transparent_b…

In [3]:
print("--- Building the LULC Exclusion Mask ---")

# 1. ESA WorldCover (The Baseline)
# 10=Trees, 20=Shrubland, 30=Grassland
worldcover = ee.ImageCollection("ESA/WorldCover/v200").filterBounds(roi).first()
lc_map = worldcover.select('Map')

is_vegetation = lc_map.eq(10).Or(lc_map.eq(20)).Or(lc_map.eq(30))
print("ESA WorldCover loaded (Keeping Trees, Shrubs, Grass).")

# 2. JRC Global Surface Water (The Pond Killer)
# If water has occurred here > 0%, flag it. We use .unmask(0) to protect dry land.
water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('occurrence').clip(roi).unmask(0)
is_dry_land = water.eq(0)
print("JRC Surface Water loaded (Excluding permanent/seasonal water).")

# 3. VIIRS Night Lights (The Concrete Killer)
# Average radiance over the 5 years. > 30 usually means streetlights/buildings.
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
    .filterBounds(roi) \
    .filterDate(START_DATE, END_DATE) \
    .select('avg_rad').mean().clip(roi)

is_dark = viirs.lt(30.0)
print("VIIRS Night Lights loaded (Excluding heavily lit urban areas).")

# 4. COMBINE
# A pixel MUST be Vegetation AND Dry Land AND Dark to survive.
# This creates an image where 1 = Valid Forest, 0 = Masked Out
valid_mask = is_vegetation.And(is_dry_land).And(is_dark).rename('Suitability_Mask')

# Calculate how much of Sanjay Van survived the bouncers
total_pixels = ee.Image.constant(1).reduceRegion(
    reducer=ee.Reducer.count(), geometry=roi, scale=10, maxPixels=1e9
).get('constant').getInfo()

valid_pixels = valid_mask.reduceRegion(
    reducer=ee.Reducer.sum(), geometry=roi, scale=10, maxPixels=1e9
).get('Suitability_Mask').getInfo()

survival_rate = (valid_pixels / total_pixels) * 100
print(f"\n--- Masking Results ---")
print(f"Total ROI Pixels (10m): {total_pixels}")
print(f"Valid Forest Pixels: {valid_pixels}")
print(f"Survival Rate: {survival_rate:.1f}% of Sanjay Van is suitable for clustering.")

# 5. VISUALIZATION (The Blackout Toggle)
Map2 = geemap.Map()
Map2.add_basemap('HYBRID')          # Google Hybrid
Map2.centerObject(roi, 14)

# We map 0 to Black (Excluded) and 1 to Neon Green (Valid)
mask_vis = {
    'min': 0,
    'max': 1,
    'palette': ['black', '00FF00']
}

Map2.addLayer(valid_mask, mask_vis, "Phase 1: Valid vs Blackout Layer")
Map2

--- Building the LULC Exclusion Mask ---
ESA WorldCover loaded (Keeping Trees, Shrubs, Grass).
JRC Surface Water loaded (Excluding permanent/seasonal water).
VIIRS Night Lights loaded (Excluding heavily lit urban areas).

--- Masking Results ---
Total ROI Pixels (10m): 148630
Valid Forest Pixels: 86422.29803921607
Survival Rate: 58.1% of Sanjay Van is suitable for clustering.


Map(center=[28.52999955882056, 77.17499999999988], controls=(WidgetControl(options=['position', 'transparent_b…

In [36]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import math

print("--- Initializing Phase 2: Interactive Harmonic Visualizer ---")

# 1. Prepare the Base Mathematical Layers (Using the Valid Mask from Phase 1)
def add_harmonics(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    time_diff = img.date().difference(ee.Date(START_DATE), 'year')
    t = ee.Image.constant(time_diff).toFloat().multiply(2 * math.pi)

    return img.addBands([
        ndvi,
        t.rename('t'),
        ee.Image.constant(1).rename('constant'),
        t.cos().rename('cos'),
        t.sin().rename('sin')
    ])

# Get cloud-filtered Sentinel-2
s2_base = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(add_harmonics)

# Compute the Harmonic Coefficients globally for the ROI
trend = s2_base.select(['constant', 't', 'cos', 'sin', 'NDVI']).reduce(ee.Reducer.linearRegression(4, 1))
coeffs = trend.select('coefficients').arrayProject([0]).arrayFlatten([['constant', 'trend', 'cos', 'sin']])

ndvi_mean = coeffs.select('constant').rename('NDVI_Mean')
ndvi_amp = coeffs.select('cos').hypot(coeffs.select('sin')).rename('NDVI_Amp')
ndvi_phase = coeffs.select('sin').atan2(coeffs.select('cos')).rename('NDVI_Phase')

# Mask to only our valid forest pixels!
harmonic_stack = ee.Image.cat([ndvi_mean, ndvi_amp, ndvi_phase]).updateMask(valid_mask)

# 2. UI Setup: Map and Output Widget
Map3 = geemap.Map()
Map3.centerObject(roi, 15)
Map3.addLayer(valid_mask, {'min': 0, 'max': 1, 'palette': ['black', '00FF00']}, "Valid Forest (Click Here)")

output_widget = widgets.Output(layout={'border': '1px solid black', 'padding': '10px'})

# 3. The Click Interaction Logic
def handle_interaction(**kwargs):
    latlon = kwargs.get('coordinates')
    if kwargs.get('type') == 'click':
        with output_widget:
            output_widget.clear_output(wait=True)
            print("Fetching 5-year time series from Google Servers... (Takes ~3-5 seconds)")

            try:
                # Create a point at the clicked location
                point = ee.Geometry.Point(latlon[::-1]) # ee needs [lon, lat]

                # Check if it's a valid pixel first
                is_valid = valid_mask.reduceRegion(ee.Reducer.first(), point, 10).get('Suitability_Mask').getInfo()
                if is_valid != 1:
                    print("You clicked a masked/blackout pixel. Please click a green forest pixel.")
                    return

                # A. Get the calculated Harmonic parameters for this specific pixel
                params = harmonic_stack.reduceRegion(ee.Reducer.first(), point, 10).getInfo()
                mean_val = params.get('NDVI_Mean')
                amp_val = params.get('NDVI_Amp')
                phase_val = params.get('NDVI_Phase')

                # B. Get the Raw Satellite Dots (Limit to 150 best points to prevent memory crashes)
                # def extract_raw(img):
                #     val = img.reduceRegion(ee.Reducer.first(), point, 10).get('NDVI')
                #     t = img.reduceRegion(ee.Reducer.first(), point, 10).get('t')
                #     return ee.Feature(None, {'NDVI': val, 't': t})

                # # Filter out nulls
                # raw_data_fc = s2_base.map(extract_raw).filter(ee.Filter.notNull(['NDVI']))
                # raw_list = raw_data_fc.toList(150).getInfo()

                # raw_t = [f['properties']['t'] for f in raw_list]
                # raw_ndvi = [f['properties']['NDVI'] for f in raw_list]

                # B. Extract ALL Raw Satellite Dots efficiently (The Gold Standard method)
                # getRegion pierces through the image stack instantly instead of mapping reducers.
                ts_data = s2_base.select(['t', 'NDVI']).getRegion(point, 10).getInfo()

                # ts_data returns a list of lists. The first row is the header.
                # Example header: ['id', 'longitude', 'latitude', 'time', 't', 'NDVI']
                header = ts_data[0]
                t_idx = header.index('t')
                ndvi_idx = header.index('NDVI')

                raw_t = []
                raw_ndvi = []

                # Loop through the data, skipping the header row
                for row in ts_data[1:]:
                    t_val = row[t_idx]
                    ndvi_val = row[ndvi_idx]

                    # Only append if the satellite actually captured a valid number (not clouded out)
                    if t_val is not None and ndvi_val is not None:
                        raw_t.append(t_val)
                        raw_ndvi.append(ndvi_val)

                print(f"Successfully loaded ALL {len(raw_t)} valid observations for this pixel!")

                # C. Plotting the Proof
                plt.figure(figsize=(10, 4))

                # Plot Raw Dots
                plt.scatter(raw_t, raw_ndvi, color='darkgreen', alpha=0.5, s=20, label='Raw Satellite Observations')

                # Generate Math Curve: Mean + Amp * cos(t - Phase)
                t_smooth = np.linspace(0, 5 * 2 * math.pi, 200) # 5 years in radians
                curve = mean_val + amp_val * np.cos(t_smooth - phase_val)

                # Plot Math Curve
                plt.plot(t_smooth, curve, color='red', linewidth=2, label='Earth Engine Harmonic Fit')

                plt.title(f"Phenology Heartbeat (Mean: {mean_val:.2f} | Amp: {amp_val:.2f} | Phase: {phase_val:.2f})")
                plt.xlabel("Time (Years represented in Radians)")
                plt.ylabel("NDVI")
                plt.ylim(0, 1)
                plt.legend(loc='upper right')
                plt.grid(True, linestyle='--', alpha=0.6)
                plt.show()

            except Exception as e:
                print(f"Error extracting data: {e}")

# 3. The Click Interaction Logic (UPGRADED FOR MENTOR DEFENSE)
def handle_interaction_comparison(**kwargs):
    latlon = kwargs.get('coordinates')
    if kwargs.get('type') == 'click':
        with output_widget:
            output_widget.clear_output(wait=True)
            print("Fetching 5-year Optical AND Radar time series... (Takes ~3-8 seconds)")

            try:
                point = ee.Geometry.Point(latlon[::-1])

                is_valid = valid_mask.reduceRegion(ee.Reducer.first(), point, 10).get('Suitability_Mask').getInfo()
                if is_valid != 1:
                    print("You clicked a masked/blackout pixel. Please click a green forest pixel.")
                    return

                # --- 1. FETCH OPTICAL (NDVI) ---
                params = harmonic_stack.reduceRegion(ee.Reducer.first(), point, 10).getInfo()
                mean_val = params.get('NDVI_Mean')
                amp_val = params.get('NDVI_Amp')
                phase_val = params.get('NDVI_Phase')

                ts_data_opt = s2_base.select(['t', 'NDVI']).getRegion(point, 10).getInfo()
                opt_header = ts_data_opt[0]
                t_idx = opt_header.index('t')
                ndvi_idx = opt_header.index('NDVI')

                raw_t = []
                raw_ndvi = []
                for row in ts_data_opt[1:]:
                    if row[t_idx] is not None and row[ndvi_idx] is not None:
                        raw_t.append(row[t_idx])
                        raw_ndvi.append(row[ndvi_idx])

                # --- 2. FETCH RADAR (S1 VH) ---
                # We pull the raw VH backscatter to prove the variance to the mentor
                s1_raw = s1_archive.select('VH').getRegion(point, 10).getInfo()
                s1_header = s1_raw[0]
                time_idx = s1_header.index('time')
                vh_idx = s1_header.index('VH')

                vh_dates = []
                vh_vals = []
                for row in s1_raw[1:]:
                    if row[time_idx] is not None and row[vh_idx] is not None:
                        # Convert ms time to years (matching our 't' scale)
                        yr = (row[time_idx] - ee.Date(START_DATE).millis().getInfo()) / (1000 * 60 * 60 * 24 * 365.25)
                        vh_dates.append(yr)
                        vh_vals.append(row[vh_idx])

                vh_min = np.percentile(vh_vals, 10)
                vh_max = np.percentile(vh_vals, 90)
                vh_var = np.std(vh_vals)

                print(f"Loaded {len(raw_t)} Optical obs and {len(vh_vals)} Radar obs.")

                # --- 3. PLOT BOTH CHARTS ---
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4))

                # Chart 1: Optical Phenology
                # Convert radians back to actual years (0 to 5)
                raw_t_years = [t / (2 * math.pi) for t in raw_t]

                # Plot the dots using the fixed years
                ax1.scatter(raw_t_years, raw_ndvi, color='darkgreen', alpha=0.5, s=20, label='Raw S2 NDVI')
                t_smooth = np.linspace(0, 5, 200) * 2 * math.pi
                curve = mean_val + amp_val * np.cos(t_smooth - phase_val)
                ax1.plot(t_smooth / (2*math.pi), curve, color='red', linewidth=2, label='Harmonic Fit')
                ax1.set_title(f"Optical (NDVI Phase & Amp)\nDNA: [{mean_val:.2f}, {amp_val:.2f}, {phase_val:.2f}]")
                ax1.set_xlabel("Years from Start")
                ax1.set_ylabel("NDVI")
                ax1.set_ylim(0, 1)
                ax1.grid(True, linestyle='--', alpha=0.6)

                # Chart 2: Radar Structure (The Proof)
                ax2.scatter(vh_dates, vh_vals, color='purple', alpha=0.4, s=15, label='Raw S1 VH')
                ax2.axhline(vh_max, color='blue', linestyle='--', label=f'Max (P90): {vh_max:.1f} dB')
                ax2.axhline(vh_min, color='brown', linestyle='--', label=f'Min (P10): {vh_min:.1f} dB')
                ax2.fill_between(vh_dates, vh_min, vh_max, color='gray', alpha=0.1)
                ax2.set_title(f"Radar (Structural Envelope)\nDNA Variance: {vh_var:.2f} dB")
                ax2.set_xlabel("Years from Start")
                ax2.set_ylabel("VH Backscatter (dB)")
                ax2.legend(loc='lower right')
                ax2.grid(True, linestyle='--', alpha=0.6)

                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"Error extracting data: {e}")

# Link the click action to the map
Map3.on_interaction(handle_interaction_comparison)

# Display the layout
display(widgets.VBox([Map3, output_widget]))
print("Dashboard Ready! Click anywhere on the Neon Green areas of the map.")

--- Initializing Phase 2: Interactive Harmonic Visualizer ---


Dashboard Ready! Click anywhere on the Neon Green areas of the map.


In [32]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import math

print("--- Initializing Phase 2: Interactive Dual-Sensor Visualizer ---")

# 1. Prepare the Base Mathematical Layers
def add_harmonics(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    time_diff = img.date().difference(ee.Date(START_DATE), 'year')
    t = ee.Image.constant(time_diff).toFloat().multiply(2 * math.pi)

    return img.addBands([
        ndvi, t.rename('t'), ee.Image.constant(1).rename('constant'),
        t.cos().rename('cos'), t.sin().rename('sin')
    ])

s2_base = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(add_harmonics)

trend = s2_base.select(['constant', 't', 'cos', 'sin', 'NDVI']).reduce(ee.Reducer.linearRegression(4, 1))
coeffs = trend.select('coefficients').arrayProject([0]).arrayFlatten([['constant', 'trend', 'cos', 'sin']])

ndvi_mean = coeffs.select('constant').rename('NDVI_Mean')
ndvi_amp = coeffs.select('cos').hypot(coeffs.select('sin')).rename('NDVI_Amp')
ndvi_phase = coeffs.select('sin').atan2(coeffs.select('cos')).rename('NDVI_Phase')
harmonic_stack = ee.Image.cat([ndvi_mean, ndvi_amp, ndvi_phase]).updateMask(valid_mask)

# 2. UI Setup: Map and Output Widget
Map3 = geemap.Map()
Map3.centerObject(roi, 15)
Map3.addLayer(valid_mask, {'min': 0, 'max': 1, 'palette': ['black', '00FF00']}, "Valid Forest (Click Here)")
output_widget = widgets.Output(layout={'border': '1px solid black', 'padding': '10px'})

# 3. The Dual-Chart Interaction Logic
def handle_interaction(**kwargs):
    latlon = kwargs.get('coordinates')
    if kwargs.get('type') == 'click':
        with output_widget:
            output_widget.clear_output(wait=True)
            print("Fetching 5-year Optical AND Radar time series... (Takes ~3-8 seconds)")

            try:
                point = ee.Geometry.Point(latlon[::-1])
                is_valid = valid_mask.reduceRegion(ee.Reducer.first(), point, 10).get('Suitability_Mask').getInfo()

                if is_valid != 1:
                    print("You clicked a masked/blackout pixel. Please click a green forest pixel.")
                    return

                # --- 1. FETCH OPTICAL (NDVI) efficiently ---
                params = harmonic_stack.reduceRegion(ee.Reducer.first(), point, 10).getInfo()
                mean_val = params.get('NDVI_Mean')
                amp_val = params.get('NDVI_Amp')
                phase_val = params.get('NDVI_Phase')

                ts_data_opt = s2_base.select(['t', 'NDVI']).getRegion(point, 10).getInfo()
                opt_header = ts_data_opt[0]
                t_idx = opt_header.index('t')
                ndvi_idx = opt_header.index('NDVI')

                raw_t = []
                raw_ndvi = []
                for row in ts_data_opt[1:]:
                    if row[t_idx] is not None and row[ndvi_idx] is not None:
                        raw_t.append(row[t_idx])
                        raw_ndvi.append(row[ndvi_idx])
                # --- NEW: THE "1-HECTARE BLUR" PROOF FOR MENTOR ---
                print("Calculating 1-Hectare Spatial Median (The 'Smooth' Line)...")
                hectare_geom = point.buffer(50) # 50m radius = roughly 100m x 100m (1 Hectare)

                def calculate_hectare_median(img):
                    # Average all 100 pixels around the click into a single smooth number
                    stats = img.select(['t', 'NDVI']).reduceRegion(
                        reducer=ee.Reducer.median(),
                        geometry=hectare_geom,
                        scale=10
                    )
                    return ee.Feature(None, {'t': stats.get('t'), 'NDVI': stats.get('NDVI')})

                hectare_fc = s2_base.map(calculate_hectare_median).filter(ee.Filter.notNull(['NDVI']))
                hectare_list = hectare_fc.getInfo()['features']

                blur_t = []
                blur_ndvi = []
                for f in hectare_list:
                    if f['properties']['t'] is not None and f['properties']['NDVI'] is not None:
                        blur_t.append(f['properties']['t'] / (2 * math.pi)) # Convert to Years
                        blur_ndvi.append(f['properties']['NDVI'])

                # --- 2. FETCH RADAR (S1 VH) efficiently ---
                s1_archive = ee.ImageCollection("COPERNICUS/S1_GRD") \
                    .filterBounds(point) \
                    .filterDate(START_DATE, END_DATE) \
                    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
                    .filter(ee.Filter.eq('instrumentMode', 'IW'))

                s1_raw = s1_archive.select('VH').getRegion(point, 10).getInfo()
                s1_header = s1_raw[0]
                time_idx = s1_header.index('time')
                vh_idx = s1_header.index('VH')

                vh_dates = []
                vh_vals = []
                for row in s1_raw[1:]:
                    if row[time_idx] is not None and row[vh_idx] is not None:
                        yr = (row[time_idx] - ee.Date(START_DATE).millis().getInfo()) / (1000 * 60 * 60 * 24 * 365.25)
                        vh_dates.append(yr)
                        vh_vals.append(row[vh_idx])

                vh_min = np.percentile(vh_vals, 10)
                vh_max = np.percentile(vh_vals, 90)
                vh_var = np.std(vh_vals)

                print(f"Loaded {len(raw_t)} Optical obs and {len(vh_vals)} Radar obs.")

                # --- 3. PLOT BOTH CHARTS SIDE-BY-SIDE ---
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4))

                # Chart 1: Optical Phenology (The Heartbeat)
                # Convert radians back to actual years (0 to 5)
                raw_t_years = [t / (2 * math.pi) for t in raw_t]

                # Plot the dots using the fixed years
                ax1.scatter(raw_t_years, raw_ndvi, color='darkgreen', alpha=0.5, s=20, label='Raw S2 NDVI')
                ax1.plot(blur_t, blur_ndvi, color='orange', linewidth=2, label='1-Hectare Median', alpha=0.8)
                t_smooth = np.linspace(0, 5, 200) * 2 * math.pi
                curve = mean_val + amp_val * np.cos(t_smooth - phase_val)
                ax1.plot(t_smooth / (2*math.pi), curve, color='red', linewidth=2, label='Harmonic Fit')
                ax1.set_title(f"Optical (NDVI Phase & Amp)\nDNA: [Mean: {mean_val:.2f}, Amp: {amp_val:.2f}, Phase: {phase_val:.2f}]")
                ax1.set_xlabel("Years from Start")
                ax1.set_ylabel("NDVI")
                ax1.set_ylim(0, 1)
                ax1.legend(loc='upper right')
                ax1.grid(True, linestyle='--', alpha=0.6)

                # Chart 2: Radar Structure (The Proof)
                ax2.scatter(vh_dates, vh_vals, color='purple', alpha=0.4, s=15, label='Raw S1 VH')
                ax2.axhline(vh_max, color='blue', linestyle='--', label=f'Max (P90): {vh_max:.1f} dB')
                ax2.axhline(vh_min, color='brown', linestyle='--', label=f'Min (P10): {vh_min:.1f} dB')
                ax2.fill_between(vh_dates, vh_min, vh_max, color='gray', alpha=0.1)
                ax2.set_title(f"Radar (Structural Envelope)\nDNA Variance: {vh_var:.2f} dB")
                ax2.set_xlabel("Years from Start")
                ax2.set_ylabel("VH Backscatter (dB)")
                ax2.legend(loc='lower right')
                ax2.grid(True, linestyle='--', alpha=0.6)

                plt.tight_layout()
                plt.show()
            except Exception as e:
                print(f"Error extracting data: {e}")

Map3.on_interaction(handle_interaction)

# This is the line that actually draws the map and the chart box on your screen!
display(widgets.VBox([Map3, output_widget]))
print("Dashboard Ready Click anywhere on the Neon Green areas of the map.")

--- Initializing Phase 2: Interactive Dual-Sensor Visualizer ---


Dashboard Ready Click anywhere on the Neon Green areas of the map.


In [30]:
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import math

print("--- Initializing Phase 2: Interactive Dual-Sensor Visualizer ---")

# 1. Prepare the Base Mathematical Layers

# ---> NEW: THE PIXEL-LEVEL CLOUD MASK <---
def mask_s2_clouds(img):
    scl = img.select('SCL')
    # SCL Classes: 3=Shadows, 8=Medium Cloud, 9=High Cloud, 10=Cirrus
    # We want to KEEP everything else (Vegetation, bare earth, etc.)
    valid_pixels = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(valid_pixels)

def add_harmonics(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    time_diff = img.date().difference(ee.Date(START_DATE), 'year')
    t = ee.Image.constant(time_diff).toFloat().multiply(2 * math.pi)

    return img.addBands([
        ndvi,
        t.rename('t'),
        ee.Image.constant(1).rename('constant'),
        t.cos().rename('cos'),
        t.sin().rename('sin')
    ])

# Get cloud-filtered Sentinel-2
s2_base = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(mask_s2_clouds) \
    .map(add_harmonics)

s2_base = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(add_harmonics)

trend = s2_base.select(['constant', 't', 'cos', 'sin', 'NDVI']).reduce(ee.Reducer.linearRegression(4, 1))
coeffs = trend.select('coefficients').arrayProject([0]).arrayFlatten([['constant', 'trend', 'cos', 'sin']])

ndvi_mean = coeffs.select('constant').rename('NDVI_Mean')
ndvi_amp = coeffs.select('cos').hypot(coeffs.select('sin')).rename('NDVI_Amp')
ndvi_phase = coeffs.select('sin').atan2(coeffs.select('cos')).rename('NDVI_Phase')
harmonic_stack = ee.Image.cat([ndvi_mean, ndvi_amp, ndvi_phase]).updateMask(valid_mask)

# 2. UI Setup: Map and Output Widget
Map3 = geemap.Map()
Map3.centerObject(roi, 15)
Map3.addLayer(valid_mask, {'min': 0, 'max': 1, 'palette': ['black', '00FF00']}, "Valid Forest (Click Here)")
output_widget = widgets.Output(layout={'border': '1px solid black', 'padding': '10px'})

# 3. The Dual-Chart Interaction Logic
# 3. The Click Interaction Logic (THE 3-PANEL MENTOR DEFENSE)
def handle_interaction(**kwargs):
    latlon = kwargs.get('coordinates')
    if kwargs.get('type') == 'click':
        with output_widget:
            output_widget.clear_output(wait=True)
            print("Fetching 10m Optical, 1-Hectare Optical, and 10m Radar data...")
            try:
                point = ee.Geometry.Point(latlon[::-1])

                is_valid = valid_mask.reduceRegion(ee.Reducer.first(), point, 10).get('Suitability_Mask').getInfo()
                if is_valid != 1:
                    print("You clicked a masked/blackout pixel. Please click a green forest pixel.")
                    return

                # --- 1. FETCH 10m OPTICAL (NDVI) ---
                params = harmonic_stack.reduceRegion(ee.Reducer.first(), point, 10).getInfo()
                mean_val = params.get('NDVI_Mean')
                amp_val = params.get('NDVI_Amp')
                phase_val = params.get('NDVI_Phase')

                ts_data_opt = s2_base.select(['t', 'NDVI']).getRegion(point, 10).getInfo()
                opt_header = ts_data_opt[0]
                t_idx = opt_header.index('t')
                ndvi_idx = opt_header.index('NDVI')

                raw_t_years = []
                raw_ndvi = []
                for row in ts_data_opt[1:]:
                    if row[t_idx] is not None and row[ndvi_idx] is not None:
                        raw_t_years.append(row[t_idx] / (2 * math.pi))
                        raw_ndvi.append(row[ndvi_idx])

                # --- 2. FETCH 1-HECTARE OPTICAL (THE SMOOTHED PROOF) ---
                hectare_geom = point.buffer(50) # 50m radius = roughly 1 hectare

                def calculate_hectare_median(img):
                    stats = img.select(['t', 'NDVI']).reduceRegion(
                        reducer=ee.Reducer.median(),
                        geometry=hectare_geom,
                        scale=10
                    )
                    return ee.Feature(None, {'t': stats.get('t'), 'NDVI': stats.get('NDVI')})

                hectare_fc = s2_base.map(calculate_hectare_median).filter(ee.Filter.notNull(['NDVI']))
                hectare_list = hectare_fc.getInfo()['features']

                blur_t_years = []
                blur_ndvi = []
                for f in hectare_list:
                    if f['properties']['t'] is not None and f['properties']['NDVI'] is not None:
                        blur_t_years.append(f['properties']['t'] / (2 * math.pi))
                        blur_ndvi.append(f['properties']['NDVI'])

                # --- 3. FETCH RADAR RATIO (VH/VV) ---
                # We pull the Ratio to show the weather-proof structural envelope
                s1_raw = s1_with_ratio.select('VH_VV_Ratio').getRegion(point, 10).getInfo()
                s1_header = s1_raw[0]
                time_idx = s1_header.index('time')
                ratio_idx = s1_header.index('VH_VV_Ratio')

                ratio_dates = []
                ratio_vals = []
                for row in s1_raw[1:]:
                    if row[time_idx] is not None and row[ratio_idx] is not None:
                        yr = (row[time_idx] - ee.Date(START_DATE).millis().getInfo()) / (1000 * 60 * 60 * 24 * 365.25)
                        ratio_dates.append(yr)
                        ratio_vals.append(row[ratio_idx])

                ratio_min = np.percentile(ratio_vals, 10)
                ratio_max = np.percentile(ratio_vals, 90)
                ratio_var = np.std(ratio_vals)

                print("Data loaded")

                # --- 4. PLOT ALL 3 CHARTS ---
                fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

                # Math Curve Generation (Used for both Optical charts)
                t_smooth_rad = np.linspace(0, 5, 200) * 2 * math.pi
                t_smooth_yr = t_smooth_rad / (2 * math.pi)
                curve = mean_val + amp_val * np.cos(t_smooth_rad - phase_val)

                # Chart 1: 10m Optical (The Messy Truth)
                ax1.scatter(raw_t_years, raw_ndvi, color='darkgreen', alpha=0.4, s=15, label='10m Raw Dots')
                ax1.plot(t_smooth_yr, curve, color='red', linewidth=2, label='10m Harmonic Fit')
                ax1.set_title("1. Spatial Purity (Our Input)\n10m Pixel: Retains Ecotone Boundaries")
                ax1.set_xlabel("Years from Start")
                ax1.set_ylabel("NDVI")
                ax1.set_ylim(0, 1)
                ax1.legend(loc='upper right')
                ax1.grid(True, linestyle='--', alpha=0.6)

                # Chart 2: 1-Hectare Optical (The Proof)
                ax2.scatter(blur_t_years, blur_ndvi, color='orange', alpha=0.8, s=20, label='1-Hectare Median Dots')
                ax2.plot(t_smooth_yr, curve, color='red', linewidth=2, label='10m Harmonic Fit')
                ax2.set_title("2. The Smoothing Proof\nSame red line perfectly fits 1-Hectare median")
                ax2.set_xlabel("Years from Start")
                ax2.set_ylim(0, 1)
                ax2.legend(loc='upper right')
                ax2.grid(True, linestyle='--', alpha=0.6)

                # Chart 3: Radar Ratio (The Structure)
                ax3.scatter(ratio_dates, ratio_vals, color='purple', alpha=0.4, s=15, label='S1 VH/VV Ratio')
                ax3.axhline(ratio_max, color='blue', linestyle='--', label=f'Peak Leaf-On (p90)')
                ax3.axhline(ratio_min, color='brown', linestyle='--', label=f'Peak Leaf-Off (p10)')
                ax3.set_title(f"3. Weather-Proof Radar\nStructural Variance: {ratio_var:.2f}")
                ax3.set_xlabel("Years from Start")
                ax3.set_ylabel("VH/VV Ratio (dB)")
                ax3.legend(loc='lower right')
                ax3.grid(True, linestyle='--', alpha=0.6)

                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"Error extracting data: {e}")

Map3.on_interaction(handle_interaction)

# This is the line that actually draws the map and the chart box on your screen!
display(widgets.VBox([Map3, output_widget]))
print("Dashboard Ready Click anywhere on the Neon Green areas of the map.")

--- Initializing Phase 2: Interactive Dual-Sensor Visualizer ---


Dashboard Ready Click anywhere on the Neon Green areas of the map.


In [14]:
print("--- Extracting the 9-Dimensional Vector ---")

# --- 1. THE HEARTBEAT (Optical / Phenology) ---
print("1. Processing Optical Harmonics (Sentinel-2)...")
trend = s2_base.select(['constant', 't', 'cos', 'sin', 'NDVI']).reduce(ee.Reducer.linearRegression(4, 1))
coeffs = trend.select('coefficients').arrayProject([0]).arrayFlatten([['constant', 'trend', 'cos', 'sin']])

ndvi_mean = coeffs.select('constant').rename('NDVI_Mean')
ndvi_amp = coeffs.select('cos').hypot(coeffs.select('sin')).rename('NDVI_Amp')
ndvi_phase = coeffs.select('sin').atan2(coeffs.select('cos')).rename('NDVI_Phase')

# --- 2. THE SKELETON (Radar / Volume) - THE WEATHER-PROOF UPGRADE ---
print("2. Processing Structural Dynamics via Polarization Ratio (Sentinel-1 SAR)...")
s1_archive = ee.ImageCollection("COPERNICUS/S1_GRD") \
    .filterBounds(roi) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('instrumentMode', 'IW'))

# Calculate the VH/VV Ratio for every single image in the 5-year stack
def calculate_ratio(image):
    # In Decibels, division is subtraction! (VH - VV)
    ratio = image.select('VH').subtract(image.select('VV')).rename('VH_VV_Ratio')
    return image.addBands(ratio)

s1_with_ratio = s1_archive.map(calculate_ratio)

# 1. Structural Limits (Using the Weather-Proof Ratio!)
# We calculate the percentiles of the RATIO, not raw VH.
s1_percentiles = s1_with_ratio.select('VH_VV_Ratio').reduce(ee.Reducer.percentile([10, 90]))

# p90 = Peak Canopy Dominance; p10 = Peak Stem/Ground Dominance
s1_max_ratio = s1_percentiles.select('VH_VV_Ratio_p90').rename('S1_Max_Ratio')
s1_min_ratio = s1_percentiles.select('VH_VV_Ratio_p10').rename('S1_Min_Ratio')

# 2. Structural Plasticity (Variance of the Ratio)
s1_variance = s1_with_ratio.select('VH_VV_Ratio').reduce(ee.Reducer.stdDev()).rename('S1_Ratio_Variance')


# --- 3. THE ROOF & STAGE (Height & Topography) ---
print("3. Processing Topography and Vertical Height...")
canopy_h = ee.Image("users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1") \
    .select('b1').rename('Canopy_Height').clip(roi).unmask(0)

dem = ee.Image("NASA/NASADEM_HGT/001").clip(roi)
elevation = dem.select('elevation').rename('Elevation')
slope = ee.Terrain.slope(elevation).rename('Slope')

# --- 4. MERGE AND MASK ---
print("4. Stacking and applying Phase 1 Bouncers...")
feature_stack = ee.Image.cat([
    ndvi_mean, ndvi_amp, ndvi_phase,   # Heartbeat
    s1_max_ratio, s1_min_ratio, s1_variance, # Skeleton (Dynamic)
    canopy_h, elevation, slope         # Roof & Stage
]).updateMask(valid_mask).toFloat()

# Print confirmation
bands = feature_stack.bandNames().getInfo()
print(f"\nFeature Stack Built Successfully: {len(bands)} Dimensions")
for i, b in enumerate(bands):
    print(f"   [{i+1}] {b}")

# --- 5. DATA INTEGRITY CHECK ---
print("\nSampling 5 valid pixels to verify math integrity...")
# Note: Using geometries=True so we can map it back to specific coordinates if needed later
sample_data = feature_stack.sample(
    region=roi,
    scale=10,
    numPixels=5,
    geometries=False
).getInfo()

import pandas as pd
df_features = pd.DataFrame([f['properties'] for f in sample_data['features']])
display(df_features.T.style.set_properties(**{'text-align': 'right'}))

--- Extracting the 9-Dimensional Vector ---
1. Processing Optical Harmonics (Sentinel-2)...
2. Processing Structural Dynamics via Polarization Ratio (Sentinel-1 SAR)...
3. Processing Topography and Vertical Height...
4. Stacking and applying Phase 1 Bouncers...

Feature Stack Built Successfully: 9 Dimensions
   [1] NDVI_Mean
   [2] NDVI_Amp
   [3] NDVI_Phase
   [4] S1_Max_Ratio
   [5] S1_Min_Ratio
   [6] S1_Ratio_Variance
   [7] Canopy_Height
   [8] Elevation
   [9] Slope

Sampling 5 valid pixels to verify math integrity...


,0,1,2
Canopy_Height,2.000000,11.000000,12.000000
Elevation,248.000000,246.000000,243.000000
NDVI_Amp,0.087809,0.018597,0.080136
NDVI_Mean,0.370144,0.559319,0.561429
NDVI_Phase,-2.917998,-1.626288,-2.460769
S1_Max_Ratio,-2.080754,-1.819422,-1.615528
S1_Min_Ratio,-10.199753,-8.427371,-9.715144
S1_Ratio_Variance,3.125524,2.680381,3.029984
Slope,7.845639,1.404842,4.208525


In [15]:
print("--- Initializing Phase 4: Normalization & SNIC ---")

def dynamic_normalize(image, region, scale=10):
    print("1. Calculating statistical variance and skewness...")

    # Calculate Skewness and Standard Deviation dynamically
    stats = image.reduceRegion(
        reducer=ee.Reducer.skew().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9
    ).getInfo()

    all_bands = image.bandNames().getInfo()
    processed_bands = []

    # Protect these from log-transformation (they handle negative/decimal math poorly)
    protected_bands = ['NDVI_Mean', 'NDVI_Amp', 'NDVI_Phase', 'S1_Max_Ratio', 'S1_Min_Ratio']

    temp_image = image

    print("2. Auto-Correcting Skewed Features...")
    for band in all_bands:
        skew = stats.get(f'{band}_skew', 0)

        # If right-skewed (> 1.0) and not protected, apply Log10(x+1)
        if skew > 1.0 and band not in protected_bands:
            print(f"   -> Log-transforming {band} (Skew: {skew:.2f})")
            # We use max(0) to prevent any negative inputs before logging
            log_band = temp_image.select(band).max(0).add(1).log10().rename(band)
            temp_image = temp_image.addBands(log_band, overwrite=True)
        else:
            print(f"   -> Keeping {band} raw (Protected or Normal Skew: {skew:.2f})")

    print("3. Z-Score Scaling all 9 dimensions...")
    # Now calculate Mean and StdDev for the Z-Score
    z_stats = temp_image.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9
    )

    def apply_z_score(band_name):
        b = ee.String(band_name)
        mean = ee.Number(z_stats.get(b.cat('_mean'), 0))
        std = ee.Number(z_stats.get(b.cat('_stdDev'), 1)).max(0.0001) # Avoid divide-by-zero
        return temp_image.select(b).subtract(mean).divide(std).rename(b)

    normalized_image = ee.ImageCollection(temp_image.bandNames().map(apply_z_score)).toBands()
    return normalized_image.rename(temp_image.bandNames())

SCALE = 10
# Execute Normalization
normalized_stack = dynamic_normalize(feature_stack, roi, SCALE)
print("9-Dimensional Stack Normalized.")

# --- RUN SNIC SEGMENTATION ---
print("\n4. Running SNIC Segmentation (Drawing the Stand Boundaries)...")
# size=10 means roughly 100m spacing for the seeds.
# compactness=0.5 balances spectral grouping with spatial continuity.
snic_seeds = ee.Algorithms.Image.Segmentation.seedGrid(10)

snic = ee.Algorithms.Image.Segmentation.SNIC(
    image=normalized_stack,
    compactness=0.5,
    connectivity=8,
    neighborhoodSize=128,
    seeds=snic_seeds
)

snic = snic.reproject(crs='EPSG:4326', scale=10)

# SNIC appends '_mean' to the band names for the object averages
object_stack = snic.select(['.*_mean'])
print("SNIC Superpixels generated. The map is now divided into ecological stands.")

# --- VISUALIZATION ---
Map4 = geemap.Map()
Map4.add_basemap('HYBRID')
Map4.centerObject(roi, 14)

# 1. The Blackout Layer
# We isolate only the '0' values from our valid mask and color them solid black
excluded_mask = valid_mask.eq(0).selfMask()
Map4.addLayer(excluded_mask, {'palette': ['black']}, "Excluded Areas (Urban/Water)")

# We visualize the random clusters to show the physical boundaries of the stands
snic_vis = snic.select('clusters').randomVisualizer().updateMask(valid_mask)
Map4.addLayer(snic_vis, {}, "Phase 4: SNIC Forest Stands")

Map4

--- Initializing Phase 4: Normalization & SNIC ---
1. Calculating statistical variance and skewness...
2. Auto-Correcting Skewed Features...
   -> Keeping NDVI_Mean raw (Protected or Normal Skew: -1.29)
   -> Keeping NDVI_Amp raw (Protected or Normal Skew: 0.45)
   -> Keeping NDVI_Phase raw (Protected or Normal Skew: 0.23)
   -> Keeping S1_Max_Ratio raw (Protected or Normal Skew: -0.80)
   -> Keeping S1_Min_Ratio raw (Protected or Normal Skew: -2.59)
   -> Log-transforming S1_Ratio_Variance (Skew: 4.37)
   -> Keeping Canopy_Height raw (Protected or Normal Skew: -0.72)
   -> Keeping Elevation raw (Protected or Normal Skew: 0.20)
   -> Keeping Slope raw (Protected or Normal Skew: 0.89)
3. Z-Score Scaling all 9 dimensions...
9-Dimensional Stack Normalized.

4. Running SNIC Segmentation (Drawing the Stand Boundaries)...
SNIC Superpixels generated. The map is now divided into ecological stands.


Map(center=[28.52999955882056, 77.17499999999988], controls=(WidgetControl(options=['position', 'transparent_b…

In [16]:
print("--- Initializing Phase 5: Ecotone Gradient Analysis ---")

# 1. Calculate the Boundary Gradients
print("1. Scanning SNIC objects for spatial gradients (calculating fences)...")
# We use a 3x3 square kernel to find the edges between different superpixels
boundary_gradients = object_stack.reduceNeighborhood(
    reducer=ee.Reducer.stdDev(),
    kernel=ee.Kernel.square(1) # 1 pixel radius = 3x3 window
)

# 2. Combine the 9-dimensional gradients into a single "Dissimilarity Score"
# Since the data was Z-scored in Phase 4, summing the standard deviations gives us a beautifully balanced Euclidean approximation.
print("2. Fusing the 9 dimensions into a Total Ecotone Score...")
ecotone_score = boundary_gradients.reduce(ee.Reducer.sum()).rename('Ecotone_Strength')

# 3. Clean up the map
# We mask out the internal pixels (where score is ~0) so we ONLY draw the borders.
# We apply a tiny threshold (0.1) to filter out floating-point math noise.
borders = ecotone_score.updateMask(ecotone_score.gt(0.1)).updateMask(valid_mask)

# --- VISUALIZATION ---
print("\n3. Rendering the Gradient Map...")
Map6 = geemap.Map()
Map6.add_basemap('HYBRID')

Map6.centerObject(roi, 15)

# Layer 1: Dark background for contrast
Map6.addLayer(valid_mask, {'min': 0, 'max': 1, 'palette': ['black', '#111111']}, "Masked Background", True)

# Layer 2: The SNIC stands (faint, so we can see the context)
snic_vis = snic.select('clusters').randomVisualizer()
Map6.addLayer(snic_vis, {'opacity': 0.3}, "SNIC Forest Stands (Context)", True)

# Layer 3: THE ECOTONE BORDERS (The Star of the Show)
# Yellow = Gentle transition (Similar stands)
# Orange = Moderate transition
# Red = HARD ECOTONE (Massive structural or species change)
ecotone_vis_params = {
    'min': 0.5,  # Minimum valid edge
    'max': 8.0,  # A score of 8+ means massive multi-dimensional shifts
    'palette': ['#ffffcc', '#fd8d3c', '#e31a1c', '#800026'] # Light Yellow to Deep Red
}

Map6.addLayer(borders, ecotone_vis_params, "Ecotone Boundaries (Red = Sharp Change)")

print("Pipeline Complete. Examine the red boundaries on your map!")
Map6

--- Initializing Phase 5: Ecotone Gradient Analysis ---
1. Scanning SNIC objects for spatial gradients (calculating fences)...
2. Fusing the 9 dimensions into a Total Ecotone Score...

3. Rendering the Gradient Map...
Pipeline Complete. Examine the red boundaries on your map!


Map(center=[28.52999955882056, 77.17499999999988], controls=(WidgetControl(options=['position', 'transparent_b…

In [17]:
# ==========================================
# PHASE 5.2: RGB Ecotone & Root Cause Analysis (TRANSPARENT FIX)
# ==========================================
print("--- Initializing Phase 5.2: RGB Boundary Analysis ---")

# 1. Calculate the standard deviation for all 9 bands individually
boundary_gradients = object_stack.reduceNeighborhood(
    reducer=ee.Reducer.stdDev(),
    kernel=ee.Kernel.square(1)
)

# 2. Split the DNA into the 3 Ecological Core Drivers
print("Isolating Phenological, Structural, and Topographic variance...")

# RED: Phenology (Species heartbeat changes)
pheno_var = boundary_gradients.select(['NDVI_Mean_mean_stdDev', 'NDVI_Amp_mean_stdDev', 'NDVI_Phase_mean_stdDev']).reduce(ee.Reducer.sum())

# GREEN: Structure (Height and radar density changes)
struct_var = boundary_gradients.select(['Canopy_Height_mean_stdDev', 'S1_Max_Ratio_mean_stdDev', 'S1_Min_Ratio_mean_stdDev', 'S1_Ratio_Variance_mean_stdDev']).reduce(ee.Reducer.sum())

# BLUE: Topography (Elevation and slope changes)
topo_var = boundary_gradients.select(['Elevation_mean_stdDev', 'Slope_mean_stdDev']).reduce(ee.Reducer.sum())

# 👇 THE FIX: Calculate total variance to use as a transparency mask 👇
total_variance = pheno_var.add(struct_var).add(topo_var)

# 3. Stack them into an RGB Image
# We apply valid_mask (to drop the city) AND total_variance.gt(0.1) to make the insides of the stands invisible!
rgb_ecotone = ee.Image.rgb(
    pheno_var.multiply(2.0),
    struct_var.multiply(1.5),
    topo_var.multiply(2.5)
).updateMask(valid_mask).updateMask(total_variance.gt(0.1))

# --- VISUALIZATION & INTERROGATION ---
Map_RGB = geemap.Map()
Map_RGB.add_basemap('HYBRID')
Map_RGB.centerObject(roi, 15)

# Layer 1: Dark background
Map_RGB.addLayer(valid_mask, {'min': 0, 'max': 1, 'palette': ['black', '#111111']}, "Dark Background", True)

# Layer 2: The SNIC stands (Now you will actually be able to see them!)
snic_vis = snic.select('clusters').randomVisualizer()
Map_RGB.addLayer(snic_vis, {'opacity': 0.4}, "SNIC Forest Stands", True)

# Layer 3: The RGB Ecotones
Map_RGB.addLayer(rgb_ecotone, {'min': 0, 'max': 3}, "RGB Ecotones (R=Species, G=Structure, B=Terrain)")

# Add the Inspector tool
Map_RGB.add_inspector()

print("Transparent RGB Map Complete. The black voids are gone!")
Map_RGB

--- Initializing Phase 5.2: RGB Boundary Analysis ---
Isolating Phenological, Structural, and Topographic variance...
Transparent RGB Map Complete. The black voids are gone!


Map(center=[28.52999955882056, 77.17499999999988], controls=(WidgetControl(options=['position', 'transparent_b…

In [18]:
print("--- Initializing Phase 5: Object-Based Clustering ---")

OPTIMAL_K = 6 # We target 6 broad ecological zones for a forest this size

# 1. Sample the Stands (Not the Pixels!)
print("1. Sampling the DNA of the SNIC stands...")
# We explicitly set scale=10 here. This locks the math, ignoring your zoom level!
training_data = object_stack.sample(
    region=roi,
    scale=SCALE,
    numPixels=5000,
    geometries=True # Keep geometries so we can map it
)

# 2. Train the Clusterer
print(f"2. Training K-Means algorithm to find {OPTIMAL_K} distinct management units...")
clusterer = ee.Clusterer.wekaKMeans(OPTIMAL_K).train(training_data)

# 3. Apply the Clustering back to the Map
print("3. Classifying the stands...")
final_management_units = object_stack.cluster(clusterer).rename('Management_Unit')

# --- MASKING THE FINAL MAP ---
# We apply our Phase 1 mask one last time to ensure crisp, clean blackout boundaries
final_map = final_management_units.updateMask(valid_mask)

# --- VISUALIZATION ---
Map5 = geemap.Map()
Map5.centerObject(roi, 14)

# Add the Blackout Background for context
Map5.addLayer(valid_mask, {'min': 0, 'max': 1, 'palette': ['black', 'black']}, "Background (Masked)", False)

# Vibrant palette to easily distinguish the 6 zones
cluster_palette = [
    '#e41a1c', # Red
    '#377eb8', # Blue
    '#4daf4a', # Green
    '#984ea3', # Purple
    '#ff7f00', # Orange
    '#ffff33'  # Yellow
]

Map5.addLayer(final_map, {'min': 0, 'max': OPTIMAL_K-1, 'palette': cluster_palette}, "Final Management Units")

print("Pipeline Complete. Rendering Map...")
Map5

--- Initializing Phase 5: Object-Based Clustering ---
1. Sampling the DNA of the SNIC stands...
2. Training K-Means algorithm to find 6 distinct management units...
3. Classifying the stands...
Pipeline Complete. Rendering Map...


Map(center=[28.52999955882056, 77.17499999999988], controls=(WidgetControl(options=['position', 'transparent_b…

In [19]:
import pandas as pd
import numpy as np

print("--- Extracting the Statistical DNA of the 6 Management Units ---")

# 1. Combine the 9-band feature stack with the 1-band cluster map
# We use the raw, un-normalized feature stack here so the numbers are physical (Meters, Decibels, etc.)
analysis_stack = feature_stack.addBands(final_management_units)

# 2. Define the bands we want to average for each cluster
band_names = feature_stack.bandNames()

# 3. Calculate the Zonal Statistics (The "DNA")
print("Calculating zonal averages... (This queries the Google servers, wait a few seconds)")
cluster_stats = analysis_stack.reduceRegion(
    reducer=ee.Reducer.mean().repeat(9).group(
        groupField=9, # The index of the 'Management_Unit' band (0-based)
        groupName='Cluster_ID'
    ),
    geometry=roi,
    scale=10,
    maxPixels=1e9
).get('groups').getInfo()

# 4. Format the Output into a Clean, Readable Table
dna_records = []
feature_names = band_names.getInfo()

for group in cluster_stats:
    c_id = int(group['Cluster_ID'])
    means = group['mean']

    # Map the means back to their original feature names
    record = {'Cluster_ID': c_id}
    for i, name in enumerate(feature_names):
        record[name] = means[i]
    dna_records.append(record)

df_dna = pd.DataFrame(dna_records).set_index('Cluster_ID').sort_index()

# Reorder columns logically: Roof -> Stage -> Heartbeat -> Skeleton
column_order = [
    'Canopy_Height', 'Elevation', 'Slope',
    'NDVI_Mean', 'NDVI_Amp', 'NDVI_Phase',
    'S1_Max_Ratio', 'S1_Min_Ratio', 'S1_Ratio_Variance'
]
df_dna = df_dna[column_order]

print("\nDNA Extraction Complete. Here is the physical profile of your forest stands:")
pd.set_option('display.float_format', '{:.3f}'.format)
display(df_dna.style.background_gradient(cmap='viridis', axis=0))

--- Extracting the Statistical DNA of the 6 Management Units ---
Calculating zonal averages... (This queries the Google servers, wait a few seconds)

DNA Extraction Complete. Here is the physical profile of your forest stands:


,Canopy_Height,Elevation,Slope,NDVI_Mean,NDVI_Amp,NDVI_Phase,S1_Max_Ratio,S1_Min_Ratio,S1_Ratio_Variance
Cluster_ID,,,,,,,,,
0,10.059309,258.868344,3.832655,0.540643,0.065788,-1.838325,-2.441085,-9.599585,2.834462
1,10.706295,242.886315,3.744326,0.531163,0.069702,-1.723514,-2.685593,-9.901032,2.857404
2,11.996281,251.983671,3.868919,0.587449,0.071925,2.265307,-2.250760,-9.225572,2.760050
3,5.568717,259.090448,4.400318,0.547156,0.109975,2.540495,-1.984935,-8.918483,2.740958
4,5.377759,255.018226,4.070608,0.473727,0.125848,-2.541380,-2.305727,-9.477221,2.858448
5,14.435743,243.401152,3.317873,0.627874,0.049608,-2.129173,-2.257488,-9.194755,2.744578


In [24]:
# ==========================================
# PHASE 5.5: THE DIAGNOSTIC STAND COMPARATOR
# ==========================================
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

print("--- Initializing Phase 5.5: 9-D Fingerprint Comparator ---")

# 1. UI Setup
Map_Compare = geemap.Map()
Map_Compare.centerObject(roi, 15)

# --- LAYER 1: The Blackout LULC Regions ---
# We isolate the '0' values from our valid_mask and color them solid black.
# Using .selfMask() ensures the forest areas become fully transparent so we can see through them.
excluded_mask = valid_mask.eq(0).selfMask()
Map_Compare.addLayer(excluded_mask, {'palette': ['black']}, "1. LULC Blackout (Urban/Water)", True)

# --- LAYER 2: The Raw SNIC Stands ---
# This visualizes the hundreds of tiny superpixels before K-Means grouped them.
# We set the 4th parameter to 'False' so it is toggled OFF by default when the map loads.
snic_vis = snic.select('clusters').randomVisualizer().updateMask(valid_mask)
Map_Compare.addLayer(snic_vis, {}, "2. SNIC Stand Boundaries", False)

# --- LAYER 3: The K-Means Management Units ---
# This is the final 6-zone map.
Map_Compare.addLayer(final_map, {'min': 0, 'max': OPTIMAL_K-1, 'palette': cluster_palette}, "3. Management Units (K-Means)", True)

# --- LAYER 4: The RGB Ecotone Fences (Optional but powerful) ---
# We can also add the red border lines we generated in Phase 5.2 as a toggle!
Map_Compare.addLayer(borders, ecotone_vis_params, "4. Ecotone Gradients (Red Borders)", False)


output_comparator = widgets.Output(layout={'border': '2px solid black', 'padding': '10px'})


# 2. State Memory (To remember the last two clicks)
clicked_stands = []

# Bands to extract
bands_raw = ['Canopy_Height_mean', 'Elevation_mean', 'Slope_mean', 'NDVI_Mean_mean', 'NDVI_Amp_mean', 'NDVI_Phase_mean', 'S1_Max_Ratio_mean', 'S1_Min_Ratio_mean', 'S1_Ratio_Variance_mean']
bands_norm = [b.replace('_mean', '') for b in bands_raw]

def create_radar_chart(stand1, stand2):
    # Setup the Radar Chart
    labels = ['Height', 'Elevation', 'Slope', 'NDVI Mean', 'NDVI Amp', 'NDVI Phase', 'Ratio Max', 'Ratio Min', 'Ratio Var']
    num_vars = len(labels)

    # Compute angle for each axis
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1] # Close the loop

    # Get Z-Scores for plotting (So all values fit on a -3 to +3 scale)
    values1 = [stand1['norm'][b] for b in bands_norm]
    values1 += values1[:1] # Close the loop

    values2 = [stand2['norm'][b] for b in bands_norm]
    values2 += values2[:1] # Close the loop

    fig, (ax_radar, ax_table) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [1, 1]})

    # --- DRAW RADAR CHART ---
    ax_radar = plt.subplot(121, polar=True)
    ax_radar.set_theta_offset(np.pi / 2)
    ax_radar.set_theta_direction(-1)
    plt.xticks(angles[:-1], labels, size=10)

    # Draw Stand A
    ax_radar.plot(angles, values1, color='blue', linewidth=2, linestyle='solid', label='Stand A (1st Click)')
    ax_radar.fill(angles, values1, color='blue', alpha=0.1)

    # Draw Stand B
    ax_radar.plot(angles, values2, color='red', linewidth=2, linestyle='solid', label='Stand B (2nd Click)')
    ax_radar.fill(angles, values2, color='red', alpha=0.1)

    ax_radar.set_title("9-Dimensional Z-Score Fingerprint\n(How K-Means sees the stands)", size=14, y=1.1)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

    # --- DRAW RAW DATA TABLE ---
    ax_table.axis('tight')
    ax_table.axis('off')

    table_data = []
    table_data.append(["Metric", "Stand A (Raw)", "Stand B (Raw)", "Absolute Diff"])
    for i, b in enumerate(bands_raw):
        val1 = stand1['raw'][b]
        val2 = stand2['raw'][b]
        diff = abs(val1 - val2)
        table_data.append([labels[i], f"{val1:.3f}", f"{val2:.3f}", f"{diff:.3f}"])

    table = ax_table.table(cellText=table_data, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.8)

    # Color the header
    for j in range(4):
        table[(0, j)].set_facecolor('#40466e')
        table[(0, j)].set_text_props(color='white', weight='bold')

    plt.tight_layout()
    plt.show()

# 3. Click Logic
def handle_comparison_click(**kwargs):
    global clicked_stands
    latlon = kwargs.get('coordinates')

    if kwargs.get('type') == 'click':
        with output_comparator:
            output_comparator.clear_output(wait=True)
            point = ee.Geometry.Point(latlon[::-1])

            # Check if valid
            is_valid = valid_mask.reduceRegion(ee.Reducer.first(), point, 10).get('Suitability_Mask').getInfo()
            if is_valid != 1:
                print("Clicked outside valid forest. Try again.")
                return

            print(f"Extracting Stand DNA... (Click {len(clicked_stands) + 1}/2)")

            # Extract Raw and Normalized DNA
            raw_dna = object_stack.reduceRegion(ee.Reducer.first(), point, 10).getInfo()
            norm_dna = normalized_stack.reduceRegion(ee.Reducer.first(), point, 10).getInfo()

            stand_data = {'raw': raw_dna, 'norm': norm_dna, 'coords': latlon}
            clicked_stands.append(stand_data)

            if len(clicked_stands) == 1:
                print("Stand A locked in. Now click a second stand (Stand B) to compare!")

            elif len(clicked_stands) == 2:
                print("Stand B locked in. Generating Fingerprint Comparison...")
                create_radar_chart(clicked_stands[0], clicked_stands[1])
                print("\nClick anywhere to reset and pick a new Stand A.")

            else:
                # Reset if they click a 3rd time
                clicked_stands = [stand_data]
                print("Reset. Stand A locked in. Click Stand B to compare.")

Map_Compare.on_interaction(handle_comparison_click)

display(widgets.VBox([Map_Compare, output_comparator]))
print("Comparator Ready! Click two stands of the SAME COLOR that are far apart to prove the clustering.")

--- Initializing Phase 5.5: 9-D Fingerprint Comparator ---


Comparator Ready! Click two stands of the SAME COLOR that are far apart to prove the clustering.


In [25]:
# ==========================================
# PHASE 5.5: THE DIAGNOSTIC STAND COMPARATOR
# ==========================================
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

print("--- Initializing Phase 5.5: 9-D Fingerprint Comparator ---")

# 1. UI Setup
Map_Compare = geemap.Map()
Map_Compare.centerObject(roi, 15)

# --- LAYER 1: The Blackout LULC Regions ---
# We isolate the '0' values from our valid_mask and color them solid black.
# Using .selfMask() ensures the forest areas become fully transparent so we can see through them.
excluded_mask = valid_mask.eq(0).selfMask()
Map_Compare.addLayer(excluded_mask, {'palette': ['black']}, "1. LULC Blackout (Urban/Water)", True)

# --- LAYER 2: The Raw SNIC Stands ---
# This visualizes the hundreds of tiny superpixels before K-Means grouped them.
# We set the 4th parameter to 'False' so it is toggled OFF by default when the map loads.
snic_vis = snic.select('clusters').randomVisualizer().updateMask(valid_mask)
Map_Compare.addLayer(snic_vis, {}, "2. SNIC Stand Boundaries", False)

# --- LAYER 3: The K-Means Management Units ---
# This is the final 6-zone map.
Map_Compare.addLayer(final_map, {'min': 0, 'max': OPTIMAL_K-1, 'palette': cluster_palette}, "3. Management Units (K-Means)", True)

# --- LAYER 4: The RGB Ecotone Fences (Optional but powerful) ---
# We can also add the red border lines we generated in Phase 5.2 as a toggle!
Map_Compare.addLayer(borders, ecotone_vis_params, "4. Ecotone Gradients (Red Borders)", False)


output_comparator = widgets.Output(layout={'border': '2px solid black', 'padding': '10px'})


# 2. State Memory (To remember the last two clicks)
clicked_stands = []

# Bands to extract
bands_raw = ['Canopy_Height_mean', 'Elevation_mean', 'Slope_mean', 'NDVI_Mean_mean', 'NDVI_Amp_mean', 'NDVI_Phase_mean', 'S1_Max_Ratio_mean', 'S1_Min_Ratio_mean', 'S1_Ratio_Variance_mean']
bands_norm = [b.replace('_mean', '') for b in bands_raw]

def create_radar_chart(stand1, stand2):
    # Setup the Radar Chart
    labels = ['Height', 'Elevation', 'Slope', 'NDVI Mean', 'NDVI Amp', 'NDVI Phase', 'Ratio Max', 'Ratio Min', 'Ratio Var']
    num_vars = len(labels)

    # Compute angle for each axis
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1] # Close the loop

    # Get Z-Scores for plotting (So all values fit on a -3 to +3 scale)
    values1 = [stand1['norm'][b] for b in bands_norm]
    values1 += values1[:1] # Close the loop

    values2 = [stand2['norm'][b] for b in bands_norm]
    values2 += values2[:1] # Close the loop

    fig, (ax_radar, ax_table) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [1, 1]})

    # --- DRAW RADAR CHART ---
    ax_radar = plt.subplot(121, polar=True)
    ax_radar.set_theta_offset(np.pi / 2)
    ax_radar.set_theta_direction(-1)
    plt.xticks(angles[:-1], labels, size=10)

    # Draw Stand A
    ax_radar.plot(angles, values1, color='blue', linewidth=2, linestyle='solid', label='Stand A (1st Click)')
    ax_radar.fill(angles, values1, color='blue', alpha=0.1)

    # Draw Stand B
    ax_radar.plot(angles, values2, color='red', linewidth=2, linestyle='solid', label='Stand B (2nd Click)')
    ax_radar.fill(angles, values2, color='red', alpha=0.1)

    ax_radar.set_title("9-Dimensional Z-Score Fingerprint\n(How K-Means sees the stands)", size=14, y=1.1)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

    # --- DRAW RAW DATA TABLE ---
    ax_table.axis('tight')
    ax_table.axis('off')

    table_data = []
    table_data.append(["Metric", "Stand A (Raw)", "Stand B (Raw)", "Absolute Diff"])
    for i, b in enumerate(bands_raw):
        val1 = stand1['raw'][b]
        val2 = stand2['raw'][b]
        diff = abs(val1 - val2)
        table_data.append([labels[i], f"{val1:.3f}", f"{val2:.3f}", f"{diff:.3f}"])

    table = ax_table.table(cellText=table_data, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.8)

    # Color the header
    for j in range(4):
        table[(0, j)].set_facecolor('#40466e')
        table[(0, j)].set_text_props(color='white', weight='bold')

    plt.tight_layout()
    plt.show()

# 3. Click Logic
def handle_comparison_click(**kwargs):
    global clicked_stands
    latlon = kwargs.get('coordinates')

    if kwargs.get('type') == 'click':
        with output_comparator:
            output_comparator.clear_output(wait=True)
            point = ee.Geometry.Point(latlon[::-1])

            # Check if valid
            is_valid = valid_mask.reduceRegion(ee.Reducer.first(), point, 10).get('Suitability_Mask').getInfo()
            if is_valid != 1:
                print("Clicked outside valid forest. Try again.")
                return

            print(f"Extracting Stand DNA... (Click {len(clicked_stands) + 1}/2)")

            # Extract Raw and Normalized DNA
            raw_dna = object_stack.reduceRegion(ee.Reducer.first(), point, 10).getInfo()
            norm_dna = normalized_stack.reduceRegion(ee.Reducer.first(), point, 10).getInfo()

            stand_data = {'raw': raw_dna, 'norm': norm_dna, 'coords': latlon}
            clicked_stands.append(stand_data)

            if len(clicked_stands) == 1:
                print("Stand A locked in. Now click a second stand (Stand B) to compare!")

            elif len(clicked_stands) == 2:
                print("Stand B locked in. Generating Fingerprint Comparison...")
                create_radar_chart(clicked_stands[0], clicked_stands[1])
                print("\nClick anywhere to reset and pick a new Stand A.")

            else:
                # Reset if they click a 3rd time
                clicked_stands = [stand_data]
                print("Reset. Stand A locked in. Click Stand B to compare.")

Map_Compare.on_interaction(handle_comparison_click)

display(widgets.VBox([Map_Compare, output_comparator]))
print("Comparator Ready! Click two stands of the SAME COLOR that are far apart to prove the clustering.")

--- Initializing Phase 5.5: 9-D Fingerprint Comparator ---


Comparator Ready! Click two stands of the SAME COLOR that are far apart to prove the clustering.


In [ ]:
print("--- Exporting Master Analysis Asset to GEE ---")

# 1. Prepare the Ecotone layers for export (renaming them so they don't clash)
rgb_export = rgb_ecotone.rename(['Ecotone_Pheno_R', 'Ecotone_Struct_G', 'Ecotone_Topo_B'])
borders_export = borders.rename(['Sharp_Boundary'])
snic_export = snic.select('clusters').rename(['SNIC_ID']).toInt32()
kmeans_export = final_management_units.rename(['KMeans_Zone']).toInt16()

# 2. Stack EVERYTHING into one super-image
master_asset = ee.Image.cat([
    feature_stack,    # The 9-D DNA (NDVI mean, amp, phase, S1 max, min, var, height, elev, slope)
    snic_export,      # The Stand Boundaries
    kmeans_export,    # The 6 Management Units
    borders_export,   # The Red Ecotone Lines
    rgb_export        # The RGB Root Cause Lines
])

# 3. Export Task
asset_id = f"projects/replicating-paper/assets/ecotone_master_sanjay_van"

task = ee.batch.Export.image.toAsset(
    image=master_asset,
    description='Export_Master_SanjayVan',
    assetId=asset_id,
    region=roi,
    scale=10,
    maxPixels=1e13
)

task.start()
print(f"Export task started! Check your GEE Tasks tab. Asset will be saved to: {asset_id}")

--- Exporting Master Analysis Asset to GEE ---
Export task started! Check your GEE Tasks tab. Asset will be saved to: projects/replicating-paper/assets/ecotone_master_sanjay_van
